In [11]:
import sys
import json
import torch
import numpy as np

from pathlib import Path

sys.path.insert(0, str(Path().resolve().parent))
from config import data_config
from config import preprocess_config

from preprocessing.preprocess_prices import preprocess_prices
from preprocessing.haar_wavelet import HaarWaveletTransform

In [12]:
"""
Raw -> Preprocessed
"""

Path("../data/preprocessed").mkdir(parents=True, exist_ok=True)
count = 0
for ticker in data_config.TICKERS:
    # Load JSON file
    with open(f'../data/raw/daily_data_{ticker}.json', 'r') as f:
        data = json.load(f)
    
    # Extract time series
    time_series = data['Time Series (Daily)']
    dates = sorted(time_series.keys())
    closing_prices = torch.tensor([float(time_series[date]['4. close']) for date in dates])
    
    # Loop through T - len(window) + 1
    ticker_samples = []
    for start in range(len(closing_prices) - preprocess_config.T):
        returns, trend, realized_vol = preprocess_prices(closing_prices, start=start)
        
        # Sample dictionary
        sample = {
            "returns": returns.tolist(),
            "trend": trend.item(),
            "realized_vol": realized_vol.item()
        }
        ticker_samples.append(sample)

        count += 1
    
    # Save to JSON file
    output_path = f'../data/preprocessed/prep_data_{ticker}.json'
    with open(output_path, 'w') as f:
        json.dump(ticker_samples, f, indent=2)
    
    print(f"{ticker}: {len(ticker_samples)} samples saved to {output_path}")

    
print(f"\nTotal samples across all tickers: {count}")


NVDA: 36 samples saved to ../data/preprocessed/prep_data_NVDA.json
GOOG: 36 samples saved to ../data/preprocessed/prep_data_GOOG.json
AAPL: 36 samples saved to ../data/preprocessed/prep_data_AAPL.json

Total samples across all tickers: 108


In [13]:
"""
Preprocessed -> Upsampled
"""

Path("../data/upsampled").mkdir(parents=True, exist_ok=True)
all_samples = []
for ticker in data_config.TICKERS:
    # Load JSON file
    with open(f'../data/preprocessed/prep_data_{ticker}.json', 'r') as f:
        all_samples.extend(json.load(f))

print(f"Total samples before upsampling: {len(all_samples)}")

# Absolute trends
abs_trends = [abs(sample['trend']) for sample in all_samples]

# Percentiles (75th, 90th, 95th)
threshold_25 = np.percentile(abs_trends, 75)
threshold_10 = np.percentile(abs_trends, 90)
threshold_5 = np.percentile(abs_trends, 95)

# Apply upsampling 
upsampled_data = []

for i in range(len(all_samples)):
    sample = all_samples[i]
    abs_trend = abs_trends[i]
    
    # Replicate according to tier
    if abs_trend >= threshold_5:
        for _ in range(5):
            upsampled_data.append(sample)
    elif abs_trend >= threshold_10:
        for _ in range(5):
            upsampled_data.append(sample)
    elif abs_trend >= threshold_25:
        for _ in range(5):
            upsampled_data.append(sample)
    else:
        upsampled_data.append(sample)

# Save to single JSON file
output_path = '../data/upsampled/upsampled_data.json'
with open(output_path, 'w') as f:
    json.dump(upsampled_data, f, indent=2)

print(f"Total samples after upsampling: {len(upsampled_data)}")

Total samples before upsampling: 108
Total samples after upsampling: 216


In [14]:
"""
Upsampled -> Train (2D)
"""

# Load JSON file
with open('../data/upsampled/upsampled_data.json', 'r') as f:
    data = json.load(f)

print(f"Loaded {len(data)} samples")

Path("../data/train").mkdir(parents=True, exist_ok=True)
transformed_data = []
for sample in data:
    # Convert returns list to torch tensor
    returns_1d = torch.tensor(sample['returns'], dtype=torch.float32)
    
    # 1D to 2D Haar Wavelet Transform
    haar_transform = HaarWaveletTransform()
    returns_2d = haar_transform.forward(returns_1d)
    
    # Nested list for easier torch.tensor conversion
    returns_2d_list = returns_2d.tolist()
    
    # Create new sample with 2D returns
    transformed_sample = {
        "returns_2d": returns_2d_list,
        "trend": sample['trend'],
        "realized_vol": sample['realized_vol']
    }
    transformed_data.append(transformed_sample)

print(f"Transformed {len(transformed_data)} samples")
print(f"Returns shape: {len(transformed_data[0]['returns_2d'])}x{len(transformed_data[0]['returns_2d'][0])}")

# Save transformed data
output_path = '../data/train/train_data_2d.json'
with open(output_path, 'w') as f:
    json.dump(transformed_data, f, indent=2)

print(f"Saved transformed data to {output_path}")
    

Loaded 216 samples
Transformed 216 samples
Returns shape: 1x8
Saved transformed data to ../data/train/train_data_2d.json
